# Predictive workflows (ISYS812)

This notebook mirrors everything under **predictive/** in one place:

- **Topic clusters** — same logic as `run_optional_nlp_topic_clusters.py` (TF-IDF + MiniBatchKMeans).
- **Escalation classifier** — same logic as `train_escalation_model.py` (text + product → formal vs other CFPB actions).

**Working directory:** the next cell resolves the project root (the folder that contains `project_paths.py`), whether your Jupyter cwd is the repo root or `predictive/`.

The authoritative narrative and caveats are in `predictive/README.txt` (also embedded at the end of this notebook).


In [5]:
from __future__ import annotations

import importlib.util
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

# --- resolve project root (repo root) ---
# Supports notebook launches from repo root, predictive/, predictive/predictive/, or deeper.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = None
for p in (_cwd, *_cwd.parents):
    if (p / "project_paths.py").is_file():
        PROJECT_ROOT = p
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find project_paths.py by walking up from current working directory. "
        "Open the notebook from this repo (or set cwd inside it)."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project_paths import PROJECT_ROOT as PR

assert PR == PROJECT_ROOT


def load_predictive_module(name: str, relative_path: str):
    """Load a script from predictive/ as a module (same top-level execution as the .py file)."""
    path = PROJECT_ROOT / relative_path
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise ImportError(path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


topics_mod = load_predictive_module("nlp_topic_clusters", "predictive/run_optional_nlp_topic_clusters.py")
esc_mod = load_predictive_module("train_escalation_model", "predictive/train_escalation_model.py")
print("Project root:", PROJECT_ROOT)
print("Loaded modules:", topics_mod.__name__, esc_mod.__name__)


Project root: /Users/asauliuk/Library/CloudStorage/OneDrive-SanFranciscoStateUniversity/ISYS812-finalProject
Loaded modules: nlp_topic_clusters train_escalation_model


## 1. Optional NLP topic clusters

Replicates `run_optional_nlp_topic_clusters.py`. Adjust `CLUSTERS`, `MAX_ROWS`, and `FORMAL_ONLY` below.

Outputs match the CLI: `output_optional_nlp_topic_clusters.csv` and `.summary.json` at the project root.


In [6]:
# --- config (CLI equivalents) ---
INPUT_CSV = PROJECT_ROOT / "master_customer_behavior_clean.csv"
OUTPUT_CSV = PROJECT_ROOT / "output_optional_nlp_topic_clusters.csv"
CLUSTERS = 20
MAX_ROWS = 50_000  # None = all rows (slow / high memory)
FORMAL_ONLY = True  # False = --all-actions

from sklearn.cluster import MiniBatchKMeans
from sklearn.feature_extraction.text import TfidfVectorizer

if not INPUT_CSV.is_file():
    raise FileNotFoundError(INPUT_CSV)

df = pd.read_csv(INPUT_CSV, low_memory=False, encoding="utf-8-sig")
if MAX_ROWS is not None:
    df = df.iloc[:MAX_ROWS].copy()

if FORMAL_ONLY and "customer_action" in df.columns:
    df = df[df["customer_action"].astype(str) == "formal_complaint"].copy()
    print(f"Rows after formal_complaint filter: {len(df):,}")
else:
    print(f"Rows: {len(df):,}")

if len(df) < CLUSTERS:
    raise ValueError("Not enough rows for this many clusters.")

df["_doc"] = df.apply(topics_mod.build_doc, axis=1)
df = df.loc[df["_doc"].str.len() > 3].copy()
print(f"Rows with usable text: {len(df):,}")

vectorizer = TfidfVectorizer(
    max_features=12_000,
    min_df=15,
    max_df=0.5,
    ngram_range=(1, 2),
    stop_words="english",
)
X = vectorizer.fit_transform(df["_doc"])
print(f"TF-IDF matrix shape: {X.shape}")

kmeans = MiniBatchKMeans(
    n_clusters=CLUSTERS,
    random_state=42,
    batch_size=4096,
    n_init=3,
)
labels = kmeans.fit_predict(X)
df["nlp_topic_id"] = labels
term_labels = topics_mod.top_terms_per_cluster(vectorizer, kmeans)
df["nlp_topic_label"] = df["nlp_topic_id"].map(term_labels)
df = df.drop(columns=["_doc"])

df.to_csv(OUTPUT_CSV, index=False)
print("Wrote:", OUTPUT_CSV)

summary = (
    df.groupby(["nlp_topic_id", "nlp_topic_label"], as_index=False)
    .size()
    .sort_values("size", ascending=False)
)
print("\nCluster sizes (top 15):\n", summary.head(15).to_string(index=False))

meta_path = OUTPUT_CSV.with_suffix(".summary.json")
payload = {
    "n_clusters": CLUSTERS,
    "n_rows": int(len(df)),
    "topics": {
        str(k): {"label": term_labels[k], "count": int((labels == k).sum())}
        for k in range(CLUSTERS)
    },
}
meta_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("Wrote topic summary:", meta_path)


Rows after formal_complaint filter: 49,189
Rows with usable text: 49,189
TF-IDF matrix shape: (49189, 1902)
Wrote: /Users/asauliuk/Library/CloudStorage/OneDrive-SanFranciscoStateUniversity/ISYS812-finalProject/output_optional_nlp_topic_clusters.csv

Cluster sizes (top 15):
  nlp_topic_id                                                                                                                     nlp_topic_label  size
            1 collection foreclosure, foreclosure, modification collection, modification, loan modification, foreclosure mortgage, mortgage, loan  6889
            6                                             debt, collect, cont attempts, collect debt, debt owed, attempts, attempts collect, owed  5406
            4                  servicing payments, payments escrow, escrow, escrow account, account mortgage, loan servicing, servicing, payments  5192
            3                     credit, information credit, incorrect information, incorrect, information, credit r

## 2. Escalation model (formal complaint vs other CFPB actions)

Replicates `train_escalation_model.py` using helpers from that module (`load_cfpb_escalation_frame`, `make_train_test`, etc.).

Tune `SPLIT`, `TRAIN_END_YEAR`, `MAX_POSITIVE_TRAIN`, and TF-IDF settings below. Writes the same artifacts as the CLI under `predictive/output_escalation_model/`.


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# --- config (CLI equivalents) ---
PATH_IN = PROJECT_ROOT / esc_mod.DEFAULT_IN
OUT_DIR = PROJECT_ROOT / "predictive/output_escalation_model"
TEST_SIZE = 0.2
RANDOM_STATE = 42
MAX_POSITIVE_TRAIN = 150_000
TFIDF_MAX_FEATURES = 12_000
TFIDF_MIN_DF = 10
SPLIT = "temporal"  # "random" | "temporal"
TRAIN_END_YEAR = 2015  # when SPLIT == "temporal"

if not PATH_IN.is_file():
    raise FileNotFoundError(PATH_IN)

df_e, y = esc_mod.load_cfpb_escalation_frame(PATH_IN)
if len(np.unique(y)) < 2:
    raise ValueError("Need both classes (formal_complaint and complaining/churning).")

X = df_e[["doc", "product_service"]]
X_train, X_test, y_train, y_test, split_meta = esc_mod.make_train_test(
    df_e,
    X,
    y,
    split=SPLIT,
    test_size=TEST_SIZE,
    train_end_year=TRAIN_END_YEAR,
    random_state=RANDOM_STATE,
)
X_fit, y_fit = esc_mod.subsample_majority_class(
    X_train,
    y_train,
    max_positive=MAX_POSITIVE_TRAIN,
    random_state=RANDOM_STATE,
)

preprocess = ColumnTransformer(
    transformers=[
        (
            "tfidf",
            TfidfVectorizer(
                max_features=TFIDF_MAX_FEATURES,
                min_df=TFIDF_MIN_DF,
                max_df=0.5,
                ngram_range=(1, 2),
                stop_words="english",
            ),
            "doc",
        ),
        (
            "product",
            OneHotEncoder(handle_unknown="ignore"),
            ["product_service"],
        ),
    ]
)
model = Pipeline(
    steps=[
        ("prep", preprocess),
        (
            "clf",
            LogisticRegression(
                solver="saga",
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)
model.fit(X_fit, y_fit)

y_score = model.predict_proba(X_test)[:, 1]
neg_s = y_score[y_test == 0]
pos_s = y_score[y_test == 1]
baseline = esc_mod.product_only_baseline(X_train, X_test, y_train, y_test, RANDOM_STATE)
metrics = {
    "n_rows_cfpb_labeled": int(len(df_e)),
    "n_train_after_subsample": int(len(X_fit)),
    "n_test": int(len(X_test)),
    "prevalence_formal_complaint_test": float(y_test.mean()),
    "split": split_meta,
    **baseline,
    "test_score_max_non_formal": float(neg_s.max()) if len(neg_s) else None,
    "test_score_min_formal": float(pos_s.min()) if len(pos_s) else None,
    "roc_auc": float(roc_auc_score(y_test, y_score)),
    "average_precision": float(average_precision_score(y_test, y_score)),
    "classification_report": classification_report(
        y_test,
        (y_score >= 0.5).astype(int),
        digits=4,
        zero_division=0,
    ),
}

OUT_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(model, OUT_DIR / "escalation_pipeline.joblib")

prep = model.named_steps["prep"]
clf = model.named_steps["clf"]
names = prep.get_feature_names_out()
pos_feats, neg_feats = esc_mod.top_coef_features(names, clf.coef_, k=45)

(OUT_DIR / "metrics.json").write_text(
    json.dumps({k: v for k, v in metrics.items() if k != "classification_report"}, indent=2),
    encoding="utf-8",
)
(OUT_DIR / "classification_report.txt").write_text(metrics["classification_report"], encoding="utf-8")
(OUT_DIR / "top_features_toward_formal_complaint.json").write_text(
    json.dumps(pos_feats, indent=2),
    encoding="utf-8",
)
(OUT_DIR / "top_features_toward_other_actions.json").write_text(
    json.dumps(neg_feats, indent=2),
    encoding="utf-8",
)

print(json.dumps({k: v for k, v in metrics.items() if k != "classification_report"}, indent=2))
print("\nClassification report (test, threshold 0.5):\n")
print(metrics["classification_report"])
print("\nWrote artifacts under:", OUT_DIR)


{
  "n_rows_cfpb_labeled": 755671,
  "n_train_after_subsample": 157128,
  "n_test": 434241,
  "prevalence_formal_complaint_test": 0.9776552651638145,
  "split": {
    "split": "temporal",
    "train_end_year": 2015,
    "train_year_min": 2014,
    "train_year_max": 2015,
    "test_year_min": 2016,
    "test_year_max": 2017
  },
  "product_only_roc_auc": 0.9177587078312577,
  "product_only_average_precision": 0.9967087502350518,
  "test_score_max_non_formal": 0.8187384331853244,
  "test_score_min_formal": 0.9314401813807965,
  "roc_auc": 1.0,
  "average_precision": 1.0
}

Classification report (test, threshold 0.5):

              precision    recall  f1-score   support

           0     1.0000    0.7466    0.8549      9703
           1     0.9942    1.0000    0.9971    424538

    accuracy                         0.9943    434241
   macro avg     0.9971    0.8733    0.9260    434241
weighted avg     0.9944    0.9943    0.9939    434241


Wrote artifacts under: /Users/asauliuk/Library/C

## 3. predictive/README.txt (embedded copy)

Below is a snapshot from the time this notebook was generated. Prefer `predictive/README.txt` on disk if it may have changed.

```text
predictive/
=============

Models and experiments that predict or score risk, themes, spikes, and escalation
patterns. Scripts assume you run them from the project root unless noted otherwise.
Paths to data are relative to the project root (see project_paths.py).


Contents (scripts)
------------------
  run_optional_nlp_topic_clusters.py
      TF-IDF + MiniBatchKMeans clusters on complaint text (optional exploration).
      Input:  master_customer_behavior_clean.csv
      Output: output_optional_nlp_topic_clusters.csv (+ .summary.json) at repo root

  train_escalation_model.py
      Supervised model: predict whether a CFPB row is tagged as a formal complaint
      versus other CFPB actions (complaining / churning), using customer-facing text
      (issue + text) and product_service.
      Input:  master_customer_behavior_clean.csv
      Output: output_escalation_model/ (under this folder) — see below


Escalation model (train_escalation_model.py)
--------------------------------------------
Driving question this script supports:
  What factors (words/themes and product line) are associated with the formal
  complaint channel versus other CFPB-tracked actions, for the same data source?

Target definition (limited by columns in the clean CSV):
  y = 1  customer_action == "formal_complaint"
  y = 0  customer_action in {"complaining", "churning"}
  Rows are restricted to source == "CFPB". The clean file does not carry post-filing
  outcomes (e.g. consumer disputed), so this is the strongest escalation-related
  label available in master_customer_behavior_clean.csv.

Features:
  - TF-IDF on a single document built from issue + text (after stripping empties)
  - One-hot encoding of product_service

Model:
  - scikit-learn Pipeline: ColumnTransformer + LogisticRegression (class_weight balanced,
    saga solver). Training can cap the majority class for speed (--max-positive-train).

Evaluation:
  - --split random   Stratified train/test holdout (default --test-size 0.2)
  - --split temporal   Train on year <= --train-end-year, test on strictly later years.
    In the current extract, CFPB rows with the three actions above span 2014–2017, so
    a useful stress test is --train-end-year 2015 (train 2014–2015, test 2016–2017).
  - metrics.json also records a product-only logistic baseline on the same split
    (product_only_roc_auc, etc.) and test_score_max_non_formal / test_score_min_formal
    to show how well scores separate classes at the ranking level.

Outputs (default: predictive/output_escalation_model/)
  escalation_pipeline.joblib     Fitted Pipeline (joblib)
  metrics.json                   Counts, split metadata, ROC-AUC, average precision,
                                 baseline metrics, optional score-range diagnostics
  classification_report.txt      Test-set report at probability threshold 0.5
  top_features_toward_formal_complaint.json
  top_features_toward_other_actions.json
                                 Largest positive / negative logistic coefficients
                                 (interpret with care: short CFPB category text can
                                 align almost deterministically with action type)

Example commands (from repo root):
  py -3 predictive/train_escalation_model.py
  py -3 predictive/train_escalation_model.py --split temporal --train-end-year 2015
  py -3 predictive/train_escalation_model.py --max-positive-train 80000


General notes
-------------
  Add notebooks and scripts here as the project grows. Keep inputs/outputs path-based
  on the project root so notebooks and Streamlit stay consistent with CLI tools.
```
